In [1]:
from faker import Faker
import pandas as pd
import random
import numpy as np

In [2]:
# Create a Faker instance with a specific locale
fake = Faker('yo_NG')  # Yoruba (Nigeria)

In [1]:
def add_inconsistencies(text, prob = 0.5):
    """Add data quality issues to text"""

    if text is None or random.random() > prob:
        return text

    ## If there is no value, just return it as-is
    ## random.random() generates a number between 0 and 1
    ## If the number is greater that 0.5, return the text unchanged
    ## If not, apply the inconsistency to the data. Only 50% of the time.
    
    issues = [
        lambda x: x.upper(), # ALL CAPS
        lambda x: x.lower(), # all lowercase # extra whitespace
        lambda x: f" {x}",
        lambda x: x.replace(' ', '  '), # double spaces
        lambda x: f"{x} ", # trailing space
        lambda x: f" {x}", # leading space
        lambda x: x.replace('.', ''), # remove periods
        lambda x: x.replace(',', '') # remove commas
    ]

    ## The above is a list of lambda functions that takes and input (x)
    ## and returns a modified version

    return random.choice(issues)(text)


In [30]:
result = add_inconsistencies("TVS Motor Company")
print(result)

TVS  Motor  Company


In [31]:
def add_phone_inconsistencies(phone):
    """Add inconsistent phone formatting"""

    if random.random() < 0.5:
        return None #Missing phone

    #Remove original formatting

    digits = ''.join(filter(str.isdigit, phone))
    ## str.isdigit is a function that checks if a character is a digit

    ## filter() goes through each character in phone and keeps only the 
    ## ones where str.isdigit isTrue

    ## .join() takes all those filtered digits and joins them into a single string

    formats = [
        digits, # just digits
        f"+234-{digits[-10:]}", #International format
        f"0{digits[-10:]}", #With leading 0
        f"{digits[-10:-7]}-{digits[-7:-4]}-{digits[-4:]}", #Dashes
        f"({digits[-10:-7]}) {digits[-7:]}", # Parentheses
        f"{digits[-10:]} ", # Trailing space
        f"234{digits[-10:]}", # Country code without +
    ]

    return random.choice(formats)

In [32]:
def add_email_inconsistencies(email):
    """Add email formatting issues"""

    if random.random() < 0.5:
        return None # Missinf=g email

    issues = [
        lambda x: x,
        lambda x: x.upper(),
        lambda x: x.lower(),
        lambda x: f" {x}",
        lambda x: f"{x} ",
        lambda x: x.replace('.', ' . '),
    ]

    return random.choice(issues)(email)


def create_duplicate_supplier(supplier, dup_type = 'exact'):
    """Create different types of duplicates"""

    dup = supplier.copy()

    if dup_type == 'exact':
        dup['supplier_id'] = f"SUP{random.randint(1, 99999):05d}"
    elif dup_type == 'similar':
        dup['supplier_id'] = f"SUP{random.randint(1, 99999):05d}"
        dup['supplier_name'] = add_inconsistencies(supplier['supplier_name'], prob = 0.8)
        dup['email'] = add_email_inconsistencies(supplier['email'])

    elif dup_type == 'partial':
        dup['supplier_id'] = f"SUP{random.randint(1, 99999):05d}"
        dup['supplier_name'] = fake.company()
        dup['address'] = fake.address()

    return dup 


In [33]:
suppliers = []
duplicate_candidates = []

In [ ]:
for i in range(500):
    supplier = {
        'supplier_id' : f'SUP{i+1:05d}',
        'supplier_name' : add_inconsistencies(fake.company()),
        'contact_person' : add_inconsistencies(fake.name()),
        'email' : add_email_inconsistencies(fake.company_email()),
        'phone' : add_phone_inconsistencies(fake.phone_number()),
        'address' : add_inconsistencies(fake.address()),
        'city' : add_inconsistencies(fake.city()),
        'state' : add_inconsistencies(fake.state()),
        'registration_date': fake.date_between(start_date='-5y', end_date='-1y')
    }

    suppliers.append(supplier)

    # Collect some suppliers for duplication
    if random.random() < 0.15:
        duplicate_candidates.append(supplier)


In [35]:
# Add duplicates (about 10% of total records)
num_duplicates = 50

for _ in range(num_duplicates):
    if duplicate_candidates:
        original = random.choice(duplicate_candidates)
        dup_type = random.choice(['exact', 'similar', 'partial'])
        duplicate = create_duplicate_supplier(original, dup_type)
        suppliers.append(duplicate)



In [36]:
# Add some completely null records (data entry errors)
for _ in range(10):
    suppliers.append({
        'supplier_id': f'SUP{random.randint(1, 99999):05d}',
        'supplier_name': None,
        'contact_person': None,
        'email': None,
        'phone': None,
        'address': None,
        'city': None,
        'state': None,
        'registration_date': None
    })

In [37]:


# Shuffle to mix duplicates throughout
random.shuffle(suppliers)

# Create datasets and save to CSV
df_suppliers = pd.DataFrame(suppliers)
df_suppliers.to_csv('suppliers.csv', index=False)

In [5]:
def generate_nigerian_phone():
    """Generate realistic Nigerian phone numbers"""

    prefixes = ['080', '081', '070', '090', '091', '071']
    prefix = random.choice(prefixes)
    suffix = ''.join([str(random.randint(0,9)) for _ in range(8)])
    return f"{prefix}{suffix}"

def add_nigerian_phone_inconsistencies(phone = None):
    """Add inconsistent Nigerian phone formatting with data quality issues"""

    # Generate a phone number if none is provided
    if phone is None:
        phone = generate_nigerian_phone()

    # 20% chance of null/missing
    if random.random() < 0.2:
        return None
    
    # Extract just the digits (in case phone already has formatting)
    digits = ''.join(filter(str.isdigit, phone))

    # Ensure we have 11 digits 
    if len(digits) < 11:
        digits = digits + ''.join([str(random.randint(0, 9)) for _ in range(11 - len(digits))])
    elif len(digits) > 11:
        digits = digits[:11]

    # Different format variations and issues

    formats = [
        digits, # 08123456789
        f"+234{digits[1:]}",
        f"234{digits[1:]}",
        f"0{digits[1:]}",
        f"{digits[:4]} {digits[4:7]} {digits[7:]}",
        f"{digits[:4]}-{digits[4:7]}-{digits[7:]}",
        f" {digits} ",
        f"{digits} ",
        f" {digits}",
        f"({digits[:4]}) {digits[4:]}",
    ]

    if random.random() < 0.9:
        non_numeric_issues = [
            f"{digits}x",
            f"0{digits[1:]}",
            f"{digits[:5]}0{digits[6:]}",
            f"{digits}-",
            f"#{digits}",
            f"{digits}.",
            f"{digits}abc"
            "N/A",
            "pending",
            "00000000000",
            f"{digits[:6]}#####{digits[6:]}".replace('#', '')
        ]

        return random.choice(formats)


In [6]:
print("Sample Nigerian phone numbers with issues.")
for _ in range(15):
    print(add_nigerian_phone_inconsistencies())

Sample Nigerian phone numbers with issues.
 09090169941 
0805-662-8395
09016288444
0800 242 7497
09038270364 
(0817) 6392347
0918-660-6875
+2348073814707
None
None
 07013183098 
(0716) 8081932
(0807) 1069347
0718-507-7455
 09073731099


In [7]:
import csv

In [9]:
rows = []
with open('suppliers.csv', 'r', encoding = 'utf-8') as f:
    reader = csv.DictReader(f)
    headers = reader.fieldnames

    for row in reader:
        # Update phone column
        row['phone'] = add_nigerian_phone_inconsistencies()
        rows.append(row)

with open('suppliers.csv', 'w', newline = '', encoding = 'utf-8') as f:
    writer = csv.DictWriter(f, fieldnames = headers)
    writer.writeheader()
    writer.writerows(rows)

print(f"Updated {len(rows)} records!")

Updated 560 records!


In [10]:
WEIGHTED_STATES = {
    'Lagos': 15,  # Commercial hub
    'Federal Capital Territory': 10,  # Capital
    'Kano': 8,
    'Rivers': 7,
    'Oyo': 6,
    'Delta': 5,
    'Kaduna': 5,
    'Ogun': 5,
    'Anambra': 4,
    'Enugu': 4,
    # Give all others weight of 1-2
    'Abia': 2, 'Adamawa': 1, 'Akwa Ibom': 3, 'Bauchi': 2, 'Bayelsa': 2,
    'Benue': 2, 'Borno': 1, 'Cross River': 2, 'Ebonyi': 1, 'Edo': 3,
    'Ekiti': 2, 'Gombe': 1, 'Imo': 3, 'Jigawa': 1, 'Katsina': 2,
    'Kebbi': 1, 'Kogi': 2, 'Kwara': 2, 'Nasarawa': 2, 'Niger': 2,
    'Ondo': 2, 'Osun': 2, 'Plateau': 2, 'Sokoto': 2, 'Taraba': 1,
    'Yobe': 1, 'Zamfara': 1
}

In [11]:
def random_weighted_state():
    """Pick a state based on weights"""
    states = list(WEIGHTED_STATES.keys())
    weights = list(WEIGHTED_STATES.values())
    return random.choices(states, weights=weights, k=1)[0]

In [12]:
def add_state_inconsistencies(state=None):
    """Add data quality issues to Nigerian states"""
    
    # If no state provided, pick a random one
    if state is None:
        state = random.choice(NIGERIAN_STATES)
    
    # 10% chance of null
    if random.random() < 0.1:
        return None
    
    # 20% chance of adding formatting issues
    if random.random() < 0.2:
        issues = [
            lambda x: x.upper(),  # "LAGOS"
            lambda x: x.lower(),  # "lagos"
            lambda x: f" {x}",    # " Lagos"
            lambda x: f"{x} ",    # "Lagos "
            lambda x: f"  {x}  ", # "  Lagos  "
            lambda x: x.replace(' ', ''),  # "FederalCapitalTerritory"
        ]
        return random.choice(issues)(state)
    
    return state

In [14]:
# Load CSV
df = pd.read_csv('suppliers.csv')

# Update with weighted random selection
df['state'] = [add_state_inconsistencies(random_weighted_state()) for _ in range(len(df))]

# Save
df.to_csv('suppliers.csv', index=False)